In [42]:
#importamos funciones para utilzar en nuestro flujo
import pandas as pd
import os
import sys
import sqlite3
import requests
import time
import kaggle
from kaggle.api.kaggle_api_extended import KaggleApi
from pathlib import Path
from dotenv import load_dotenv #pip install python-dotenv
from googleapiclient.discovery import build
from google.oauth2 import service_account #pip install google-auth
from googleapiclient.http import MediaFileUpload
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from mimetypes import guess_type
import hashlib
import json
from datetime import datetime


ruta_utils = os.path.expanduser('~/Documents/IA/4_semestre/ing_de_datos/ing_datos/tools')
sys.path.append(ruta_utils)

import tools

#from tools import DRIVE_IDS
print(f"Tools importado correctamente")
print(f"Los métodos alojados en las herramientas son {dir(tools)}")

Tools importado correctamente
Los métodos alojados en las herramientas son ['BASE_DIR', 'DRIVE_IDS', 'InstalledAppFlow', 'KAGGLE_JSON_PATH', 'KaggleApi', 'MediaFileUpload', 'Path', 'RUTA_CARPETA_BASES_KAGGLE', 'RUTA_CARPETA_BASE_SQL', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'build', 'drive', 'glob', 'guess_type', 'kaggle', 'load_dotenv', 'os', 'pd', 'requests', 'service_account', 'sql', 'sqlite3', 'sys', 'tempfile', 'time']


In [63]:
# ----------- opcional no afecta si deciden colocar el nombre la tabla -----------
TABLA_RAW       = 'data'          # origen
TABLA_EXTRACTED = 'extracted_data'    # destino




# ----------- Carpetas de esta capa -----------
PROYECTO_BASE = '/Users/memoblack1/Documents/IA/4_semestre/ing_de_datos/ing_datos'
EXTRACCION_PATH = f'{PROYECTO_BASE}/extraccion_datos'
LOGS_PATH       = f'{EXTRACCION_PATH}/Logs'
EXPORTS_PATH    = f'{EXTRACCION_PATH}/Exportaciones'

# ----------- Archivos -----------

CDC_PATH = f'{LOGS_PATH}/cdc_control.json'        # estado del control de cambios
LOG_PATH = f'{LOGS_PATH}/extraccion_log.json'    # historial de operaciones


# -----------  Si no existen, crearlos -----------
os.makedirs(LOGS_PATH, exist_ok=True)
os.makedirs(EXPORTS_PATH, exist_ok=True)




print()
print(f'Origen  : {TABLA_RAW}')
print(f'Destino : {TABLA_EXTRACTED}')
print(f'Logs    : {LOGS_PATH}')
print()


Origen  : data
Destino : extracted_data
Logs    : /Users/memoblack1/Documents/IA/4_semestre/ing_de_datos/ing_datos/extraccion_datos/Logs



In [64]:
DB_PATH = f'{PROYECTO_BASE}/bases/tabla_1.db'
TABLA_RAW = 'data'
def listar_tablas(db_path=None):

    """
    Lista todas las tablas del archivo .db clasificadas por capa
    (raw_, extracted_, transformed_, serving_).

    Parámetros:
        db_path : ruta al archivo .db (usa DB_PATH por defecto)

    Retorna:
        Lista con los nombres de las tablas encontradas
    """

    if db_path is None:
        db_path = DB_PATH

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
    tablas = [row[0] for row in cursor.fetchall()]

    print(f'\n  Tablas en: {os.path.basename(db_path)}\n')
    print(f'  {"TABLA":<35} {"CAPA":<15} {"FILAS":>8} {"COLS":>6}')
    print(f'  {"-"*35} {"-"*15} {"-"*8} {"-"*6}')

    for tabla in tablas:
        df_t = pd.read_sql_query(f'SELECT * FROM "{tabla}"', conn)
        capa = tabla.split('_')[0] + '_' if '_' in tabla else 'otra'
        print(f'  {tabla:<35} {capa:<15} {df_t.shape[0]:>8,} {df_t.shape[1]:>6}')

    conn.close()
    return tablas

listar_tablas(DB_PATH)


  Tablas en: tabla_1.db

  TABLA                               CAPA               FILAS   COLS
  ----------------------------------- --------------- -------- ------
  data                                otra             151,445     11
  extracted_data                      extracted_       151,445     15


['data', 'extracted_data']

In [65]:
def inspeccionar_dataset(df, nombre='Dataset'):

    """

    Muestra un perfil básico de un DataFrame:
    dimensiones, tipos de datos, valores nulos y primeras filas.

    Parámetros:

        df     : DataFrame a inspeccionar
        nombre : etiqueta descriptiva para el encabezado

    """


    print(f'\n{"=" * 55}')
    print(f'  DATASET: {nombre}')
    print(f'{"=" * 55}')
    print(f'  Filas    : {df.shape[0]:,}')
    print(f'  Columnas : {df.shape[1]}')
    print(f'\n  Tipos de datos y valores nulos:')

    for col, dtype in df.dtypes.items():
        nulos = df[col].isna().sum()
        pct = nulos / len(df) * 100
        print(f'    {col:<28} {str(dtype):<12} nulos: {nulos} ({pct:.1f}%)')

    print(f'\n  Primeras 3 filas:')
    try:
        from IPython.display import display
        display(df.head(3))
    except ImportError:
        print(df.head(3).to_string())
    print()
# ── Leer raw_df1 en modo solo lectura ──────────────────────────────────────
conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)

df_raw = pd.read_sql_query(f'SELECT * FROM {TABLA_RAW}', conn)
conn.close()

inspeccionar_dataset(df_raw, nombre='data')


  DATASET: data
  Filas    : 151,445
  Columnas : 11

  Tipos de datos y valores nulos:
    work_year                    int64        nulos: 0 (0.0%)
    experience_level             object       nulos: 0 (0.0%)
    employment_type              object       nulos: 0 (0.0%)
    job_title                    object       nulos: 0 (0.0%)
    salary                       int64        nulos: 0 (0.0%)
    salary_currency              object       nulos: 0 (0.0%)
    salary_in_usd                int64        nulos: 0 (0.0%)
    employee_residence           object       nulos: 0 (0.0%)
    remote_ratio                 int64        nulos: 0 (0.0%)
    company_location             object       nulos: 0 (0.0%)
    company_size                 object       nulos: 0 (0.0%)

  Primeras 3 filas:


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2025,EX,FT,Head of Data,348516,USD,348516,US,0,US,M
1,2025,EX,FT,Head of Data,232344,USD,232344,US,0,US,M
2,2025,SE,FT,Data Scientist,145400,USD,145400,US,0,US,M


In [66]:
def calcular_huella(df):

    '''Regresa la huella
    (16 caracteres) de cada
    fila del DataFrame.'''

    texto_fila = df.map(str).agg('|'.join, axis=1)              # 'v1|v2|v3|...'
    return texto_fila.map(lambda t: hashlib.sha256(t.encode('utf-8')).hexdigest()[:16])

In [67]:
prueba = df_raw.head(3).copy()
prueba['_row_hash'] = calcular_huella(df_raw.head(3))
prueba

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size,_row_hash
0,2025,EX,FT,Head of Data,348516,USD,348516,US,0,US,M,95b566db6a71c35f
1,2025,EX,FT,Head of Data,232344,USD,232344,US,0,US,M,d3a6b1cd671b1e99
2,2025,SE,FT,Data Scientist,145400,USD,145400,US,0,US,M,7150c9650f025496


In [68]:
# ── Funciones de control de cambios y log ──────────────────────────────────

# ── Leer cdc ──────────────────────────────────
def leer_cdc():

    '''Lee el estado del control de cambios (cdc_control.json)
       vacío si es la primera vez '''


    if not os.path.exists(CDC_PATH):
        return {}
    with open(CDC_PATH, encoding='utf-8') as f:
        return json.load(f)


# ── Guardar cdc ──────────────────────────────────

def guardar_cdc(estado):
    with open(CDC_PATH, 'w', encoding='utf-8') as f:
        json.dump(estado, f, indent=2, ensure_ascii=False)



# ── Escribe cdc ──────────────────────────────────

def escribir_log(filas_leidas, filas_nuevas, tipo, estado='exitoso', error=None):

    '''Agrega UNA línea al log (formato JSON Lines: un JSON por línea).'''

    evento = {
        'timestamp'    : datetime.now().isoformat(timespec='seconds'),
        'tabla_origen' : TABLA_RAW,
        'tabla_destino': TABLA_EXTRACTED,
        'filas_leidas' : filas_leidas,
        'filas_nuevas' : filas_nuevas,
        'tipo'         : tipo,
        'estado'       : estado,
        'error'        : error,
    }

    with open(LOG_PATH, 'a', encoding='utf-8') as f:     # 'a' = agregar al final
        f.write(json.dumps(evento, ensure_ascii=False) + '\n')






# ── Historial ──────────────────────────────────

def ver_log():
    if not os.path.exists(LOG_PATH):
        print('Todavía no hay operaciones registradas.')
        return pd.DataFrame()
    return pd.read_json(LOG_PATH, lines=True)




print()
print('Funciones listas')
print()


Funciones listas



In [69]:
print(leer_cdc())

{}


In [70]:
# ── La función de extracción ───────────────────────────────────────────────

def extraer():

    '''Extrae raw_df1 → extracted_df1, agregando solo las filas nuevas.'''


    try:

        # 1. Leer el origen (solo lectura)
        conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)
        df = pd.read_sql_query(f'SELECT * FROM {TABLA_RAW}', conn)



        # 2. Huellas que ya están en EXTRACTED (si la tabla existe)

        existe = conn.execute(
            "SELECT COUNT(*) FROM sqlite_master WHERE type = 'table' AND name = ?",
            (TABLA_EXTRACTED,)
        ).fetchone()[0] > 0


        if existe:
            previas = set(pd.read_sql_query(f'SELECT _row_hash FROM {TABLA_EXTRACTED}', conn)['_row_hash'])
        else:
            previas = set()
        conn.close()



        # 3. Calcular la huella de cada fila y quedarse solo con las nuevas

        df['_row_hash'] = calcular_huella(df)
        nuevas = df[~df['_row_hash'].isin(previas)].copy()

        tipo = 'full_load' if not previas else 'incremental'
        cdc = leer_cdc()




        # 4. Si hay filas nuevas: metadatos + AGREGAR (append) a extracted_df1

        if len(nuevas) > 0:

            batch_id = cdc.get('ultimo_batch', 0) + 1

            nuevas['_batch_id']     = batch_id
            nuevas['_extracted_at'] = datetime.now().isoformat(timespec='seconds')
            nuevas['_source_table'] = TABLA_RAW


            # Orden: columnas originales y al final los metadatos
            columnas = [c for c in nuevas.columns if not c.startswith('_')]
            nuevas = nuevas[columnas + ['_batch_id', '_extracted_at', '_source_table', '_row_hash']]

            conn = sqlite3.connect(DB_PATH)                      # lectura y escritura
            nuevas.to_sql(TABLA_EXTRACTED, conn, if_exists='append', index=False)
            conn.commit()
            conn.close()

            cdc['ultimo_batch'] = batch_id



        # 5. Actualizar control de cambios y log

        cdc.update({

            'tabla_origen'     : TABLA_RAW,
            'tabla_destino'    : TABLA_EXTRACTED,
            'ultima_extraccion': datetime.now().isoformat(timespec='seconds'),
            'filas_leidas'     : len(df),
            'filas_nuevas'     : len(nuevas),
            'filas_acumuladas' : len(previas) + len(nuevas),
            'tipo'             : tipo,
        })


        guardar_cdc(cdc)



        escribir_log(len(df), len(nuevas), tipo)

        print()
        print(f'{TABLA_RAW} → {TABLA_EXTRACTED}  ({tipo})')
        print(f'  Filas leídas : {len(df):,}')
        print(f'  Filas nuevas : {len(nuevas):,}')
        print()
        return nuevas



    except Exception as e:
        escribir_log(0, 0, 'error', estado='fallido', error=str(e))
        raise          # muestra el error en pantalla


print()
print('extraer() definida')
print()


extraer() definida



In [71]:
df_nuevas = extraer()


data → extracted_data  (incremental)
  Filas leídas : 151,445
  Filas nuevas : 0



In [73]:
# ── Ver la tabla EXTRACTED ─────────────────────────────────────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)

display(pd.read_sql_query(f'SELECT * FROM {TABLA_EXTRACTED} LIMIT 5', conn))

conn.close()

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size,_batch_id,_extracted_at,_source_table,_row_hash
0,2025,EX,FT,Head of Data,348516,USD,348516,US,0,US,M,1,2026-10-22T20:10:18,data,95b566db6a71c35f
1,2025,EX,FT,Head of Data,232344,USD,232344,US,0,US,M,1,2026-10-22T20:10:18,data,d3a6b1cd671b1e99
2,2025,SE,FT,Data Scientist,145400,USD,145400,US,0,US,M,1,2026-10-22T20:10:18,data,7150c9650f025496
3,2025,SE,FT,Data Scientist,81600,USD,81600,US,0,US,M,1,2026-10-22T20:10:18,data,4ff664fae1d58f98
4,2025,MI,FT,Engineer,160000,USD,160000,US,100,US,M,1,2026-10-22T20:10:18,data,ecf9cbea6e0b1327


In [74]:
print(json.dumps(leer_cdc(), indent=2, ensure_ascii=False))

{
  "tabla_origen": "data",
  "tabla_destino": "extracted_data",
  "ultima_extraccion": "2026-10-22T20:19:56",
  "filas_leidas": 151445,
  "filas_nuevas": 0,
  "filas_acumuladas": 71913,
  "tipo": "incremental"
}


In [75]:
# ── Log de operaciones ─────────────────────────────────────────────────────

ver_log()

,timestamp,tabla_origen,tabla_destino,filas_leidas,filas_nuevas,tipo,estado,error
0,2026-10-22 20:19:11,data,extracted_data,151445,0,incremental,exitoso,NaN
1,2026-10-22 20:19:56,data,extracted_data,151445,0,incremental,exitoso,NaN
